In [ ]:
import pandas as pd

INPUT_FILE = "databento.csv"
OUTPUT_FILE = "nq_final.csv"

print("Starting streaming build of continuous contract...")

writer_initialized = False

buffer = []

# =====================================================
# STREAM DATA IN CHUNKS
# =====================================================

for i, chunk in enumerate(pd.read_csv(
    INPUT_FILE,
    usecols=[
        "ts_event",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ],
    chunksize=chunksize
)):
    print(f"Processing chunk {i+1}")

    # Convert timestamp
    chunk["datetime"] = pd.to_datetime(
        chunk["ts_event"],
        utc=True
    )

    chunk.drop(columns=["ts_event"], inplace=True)

    # Remove spreads immediately
    chunk = chunk[~chunk["symbol"].str.contains("-", regex=False)]

    # Keep only NQ contracts
    chunk = chunk[chunk["symbol"].str.startswith("NQ")]

    buffer.append(chunk)

# =====================================================
# COMBINE ALL CHUNKS
# =====================================================

print("Combining chunks...")

df = pd.concat(buffer, ignore_index=True)

del buffer  # free memory

print("Total rows after load:", len(df))

# =====================================================
# SORT FOR TIME SERIES CONSISTENCY
# =====================================================

df = df.sort_values(["datetime", "symbol"])

# =====================================================
# BUILD CONTINUOUS CONTRACT
# =====================================================
# At each timestamp:
# - multiple contracts exist
# - choose highest volume contract (standard heuristic)

continuous = (
    df.sort_values(["datetime", "volume"], ascending=[True, False])
      .groupby("datetime", as_index=False)
      .first()
)

# =====================================================
# OPTIONAL: CLEAN FINAL OUTPUT
# =====================================================

continuous = continuous[
    [
        "datetime",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
]

# =====================================================
# SAVE RESULT
# =====================================================

print("Saving output...")

continuous.to_csv(OUTPUT_FILE, index=False)

print("\nDONE")
print("Output file:", OUTPUT_FILE)
print("Rows:", len(continuous))

print("\nContract distribution:")
print(continuous["symbol"].value_counts())